In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor

train = pd.read_csv('home-data-for-ml-course/train.csv')
test = pd.read_csv('home-data-for-ml-course/test.csv')

y = np.log1p(train["SalePrice"])  # log transform
X = train.drop("SalePrice", axis=1)
X_test = test.copy()


X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)


cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if X[c].dtype != "object"]


preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]), cat_cols)
    ]
)


model = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_depth=10,
    max_iter=800,
    min_samples_leaf=20,
    l2_regularization=0.01,
    random_state=42
)

pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

val_preds = pipeline.predict(X_valid)
val_preds = np.expm1(val_preds)  # reverse log transform
mae = mean_absolute_error(np.expm1(y_valid), val_preds)

print(f" Validation MAE: {mae:.2f}")


pipeline.fit(X, y)

test_preds = pipeline.predict(X_test)
test_preds = np.expm1(test_preds)  # reverse log transform

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_preds
})

submission.to_csv("submission.csv", index=False)
print(" submission.csv created")
submission.head()

 Validation MAE: 16832.77
 submission.csv created


,Id,SalePrice
0,1461,118653.710846
1,1462,152866.649942
2,1463,186815.198539
3,1464,188887.601527
4,1465,186920.179234


In [2]:
submission.to_csv('submission2.4.csv', index=False)